[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_07/TSA_ch7_pairs_trading/TSA_ch7_pairs_trading.ipynb)

# TSA_ch7_pairs_trading

Pairs trading with cointegration: Banca Transilvania and BRD since 2014 (full-sample spread, half-life); a rolling rule without look-ahead (252-day formation with Engle-Granger selection, 126-day trading, entry at |z| > 2, exit at 0) on 8 liquid BVB shares and on 6 US banks, gross and net of transaction costs, with the break-even cost and the in-sample illusion of full-sample parameters.

*Time Series Analysis (TSA), Chapter 7: Cointegration and VECM. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_7'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (Phillips-Ouliaris test) is not preinstalled in Google Colab: pip install arch
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # default size about 7 inches wide (as in MFM); a chart is shown 11-13 cm wide on a slide, so these font sizes
    # give 7-9 pt text there. Multi-panel charts should stay near 7-7.6 inches wide: at 10-11 inches the axis
    # text drops to about 5 pt on the slide.
    rc['figure.figsize'] = (7.0, 3.4)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)
    _remember_tight_layout()


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import itertools

import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller, coint, acf

from statsmodels.tsa.adfvalues import mackinnoncrit, mackinnonp

from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_order

from statsmodels.tsa.api import VAR

from arch.unitroot.cointegration import phillips_ouliaris

SEED = 2026

YIELDS = ['GS1', 'GS5', 'GS10']

YIELD_START = '1960-01-01'

HICP_RO = ('prc_hicp_minr', 'M.I15.TOTAL.RO')

HICP_EA = ('prc_hicp_minr', 'M.I15.TOTAL.EA')

ROBOR = ('irt_st_m', 'M.IRT_M3.RO')

EURIBOR = ('irt_st_m', 'M.IRT_M3.EA')

CONS_RO = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.P31_S14.RO')

GDP_RO = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')

BVB = ['TLV.RO', 'BRD.RO', 'SNP.RO', 'TGN.RO', 'TEL.RO', 'EL.RO', 'SNG.RO', 'SNN.RO']

US_BANKS = ['JPM.US', 'BAC.US', 'C.US', 'WFC.US', 'GS.US', 'MS.US']

PAIR = ['TLV.RO', 'BRD.RO']

PAIR_START = '2014-01-01'

FORM, TRADE, ENTRY = 252, 126, 2.0

COST_BVB, COST_US = 0.002, 0.0005

def us_yields(cols=YIELDS, start=YIELD_START):
    """US Treasury constant-maturity yields (FRED, monthly averages, % per year), monthly PeriodIndex."""
    f = read_fred(cols).dropna().loc[start:]
    f.index = pd.PeriodIndex(f.index, freq='M')
    return f


def us_macro():
    """US quarterly real GDP, consumption and investment (statsmodels macrodata, 1959Q1-2009Q3), 100 ln."""
    d = load_statsmodels('macrodata')
    return 100 * np.log(d[['realgdp', 'realcons', 'realinv']]).rename(
        columns={'realgdp': 'y', 'realcons': 'c', 'realinv': 'i'})


def cee_fx(start='2005-07-01'):
    """EUR/RON, EUR/HUF and EUR/PLN (month-end, from the BNR reference rates: EUR/HUF = EUR/RON / HUF/RON), 100 ln."""
    e, h, p = (read_reference_rate(c, start=start) for c in ('EUR', 'HUF', 'PLN'))
    d = pd.concat([e, h, p], axis=1, keys=['eur', 'huf', 'pln']).dropna()
    fx = pd.DataFrame({'EUR/RON': d.eur, 'EUR/HUF': 100 * d.eur / d.huf, 'EUR/PLN': d.eur / d.pln})
    return 100 * np.log(fx.resample('ME').last())


def stock(symbol, start):
    """Adjusted close of a stock from data/market (weekdays)."""
    t = read_market(symbol)
    s = pd.to_numeric(t['adjusted_close'], errors='coerce').loc[start:'2026-09-18'].dropna()
    s = s[s > 0]
    return s[s.index.dayofweek < 5].rename(symbol.split('.')[0])


def stock_panel(symbols, start, calendar='bet'):
    """Several stocks on a common calendar (BET or S&P 500 trading days); a missing quote is carried forward
    for at most 5 days (no trade that day), then the rows with any missing value are dropped."""
    cal = load_close(calendar, start=start).index
    P = pd.concat([stock(s, start) for s in symbols], axis=1).reindex(cal).ffill(limit=5)
    return P.dropna()


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def years_axis(ax, step=5):
    import matplotlib.dates as mdates
    ax.xaxis.set_major_locator(mdates.YearLocator(step))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


def ts_index(x):
    """A DatetimeIndex for plotting (PeriodIndex -> timestamps)."""
    return x.index.to_timestamp() if isinstance(x.index, pd.PeriodIndex) else x.index


def adf_test(x, reg='c'):
    """ADF test (lags by AIC): statistic, MacKinnon p-value, lags, 5% critical value."""
    r = adfuller(np.asarray(pd.Series(x).dropna(), float), regression=reg, autolag='AIC')
    return {'stat': float(r[0]), 'p': float(r[1]), 'lags': int(r[2]), 'crit5': float(r[4]['5%'])}


def ols(y, X):
    """OLS with a constant: coefficients, standard errors, t-statistics, residuals, R^2, DW."""
    m = sm.OLS(np.asarray(y, float), sm.add_constant(np.asarray(X, float))).fit()
    e = m.resid
    return {'b': m.params.tolist(), 'se': m.bse.tolist(), 't': m.tvalues.tolist(), 'r2': float(m.rsquared),
            'dw': float(np.sum(np.diff(e) ** 2) / np.sum(e ** 2)), 'resid': e, 'n': int(m.nobs)}


def eg_test(y, x, trend='c'):
    """Engle-Granger two-step test: OLS of y on a constant and x (step 1), ADF on the residuals with lags by AIC
    and MacKinnon (2010) p-values for len(x)+1 variables (step 2); Phillips-Ouliaris Z_t on the same regression."""
    y = pd.Series(y)
    X = pd.DataFrame(x)
    d = pd.concat([y, X], axis=1).dropna()
    yv, Xv = d.iloc[:, 0], d.iloc[:, 1:]
    stat, p, cv = coint(yv, Xv, trend=trend, autolag='aic')
    step1 = ols(yv, Xv)
    po = phillips_ouliaris(yv, Xv, trend=trend, test_type='Zt')
    lags = adfuller(step1['resid'], regression='n', autolag='AIC')[2]
    return {'n': int(len(d)), 'beta': [float(b) for b in step1['b'][1:]], 'const': float(step1['b'][0]),
            'r2': step1['r2'], 'dw': step1['dw'], 'tau': float(stat), 'p': float(p), 'crit5': float(cv[1]),
            'crit1': float(cv[0]), 'crit10': float(cv[2]), 'lags': int(lags), 'po_zt': float(po.stat),
            'po_p': float(po.pvalue), 'po_crit5': float(po.critical_values[5]),
            'adf_crit5': float(mackinnoncrit(N=1, regression='c', nobs=len(d))[1])}


def half_life(rho):
    """Half-life (periods) of a deviation that decays like rho^h: ln(0.5)/ln(rho), for 0 < rho < 1."""
    return float(np.log(0.5) / np.log(rho)) if 0 < rho < 1 else float('inf')


def johansen(df, det_order=0, k_ar_diff=None, maxlags=8):
    """Johansen trace and maximum-eigenvalue tests (MacKinnon-Haug-Michelis critical values via statsmodels).
    k_ar_diff = lags of the differences in the VECM, by BIC on the levels VAR if None. The rank is the first r whose
    null is not rejected at 5% in the sequence r = 0, 1, ..."""
    x = df.dropna()
    if k_ar_diff is None:
        k_ar_diff = int(max(select_order(x, maxlags=maxlags, deterministic='co').bic, 1))
    r = coint_johansen(x, det_order, k_ar_diff)
    k = x.shape[1]
    rank = lambda s, c: next((i for i in range(k) if s[i] < c[i, 1]), k)
    return {'n': int(len(x)), 'k_ar_diff': k_ar_diff, 'eig': r.eig.tolist(), 'trace': r.lr1.tolist(),
            'trace_cv5': r.cvt[:, 1].tolist(), 'maxeig': r.lr2.tolist(), 'maxeig_cv5': r.cvm[:, 1].tolist(),
            'rank_trace': rank(r.lr1, r.cvt), 'rank_maxeig': rank(r.lr2, r.cvm), 'first': str(x.index[0])[:10],
            'last': str(x.index[-1])[:10]}


def ecm_fit(y, x, lags=1):
    """Single-equation ECM (two steps): e_t = y_t - a - b x_t from the levels regression, then
    Delta y_t = c + gamma e_{t-1} + delta_0 Delta x_t + sum_j (phi_j Delta y_{t-j} + delta_j Delta x_{t-j}) + eps_t."""
    d = pd.concat([pd.Series(y, name='y'), pd.Series(x, name='x')], axis=1).dropna()
    s1 = ols(d.y, d.x)
    e = pd.Series(s1['resid'], index=d.index)
    Z = pd.DataFrame({'e1': e.shift(1), 'dx': d.x.diff()})
    for j in range(1, lags + 1):
        Z[f'dy{j}'] = d.y.diff().shift(j)
        Z[f'dx{j}'] = d.x.diff().shift(j)
    Z['dy'] = d.y.diff()
    Z = Z.dropna()
    m = sm.OLS(Z.dy, sm.add_constant(Z.drop(columns='dy'))).fit()
    g = float(m.params['e1'])
    return {'beta': float(s1['b'][1]), 'a': float(s1['b'][0]), 'gamma': g, 'gamma_se': float(m.bse['e1']),
            'gamma_t': float(m.tvalues['e1']), 'delta0': float(m.params['dx']), 'delta0_se': float(m.bse['dx']),
            'const': float(m.params['const']), 'r2': float(m.rsquared), 'n': int(m.nobs), 'half': half_life(1 + g),
            'params': {k: float(v) for k, v in m.params.items()}, 'tvalues': {k: float(v) for k, v in m.tvalues.items()}}


def pairs_backtest(P, form=FORM, trade=TRADE, entry=ENTRY, cost=0.0, pmax=0.05):
    """Pairs trading with rolling windows (no look-ahead). Every `trade` days: on the previous `form` days, test each
    pair (log prices, Engle-Granger with a constant); for every pair with p < pmax, keep the OLS hedge ratio b and
    the mean and standard deviation of the spread s = ln A - a - b ln B. In the next `trade` days: open a short
    spread position (short A, long b of B) when z > entry, a long one when z < -entry; close when z crosses 0 or at
    the end of the window. Daily return of a position: pos (r_A - b r_B) / (1 + |b|), per unit of gross exposure;
    cost = cost per unit of gross exposure traded (charged when opening and when closing). The capital is split
    equally between the pairs selected in a window."""
    L = np.log(P)
    R = L.diff()
    names = list(P.columns)
    days, gross, net, n_sel, trades, wins, pair_ret = [], [], [], [], 0, 0, []
    for s0 in range(form, len(P) - 1, trade):
        fw = L.iloc[s0 - form:s0]
        sel = []
        for a, b in itertools.combinations(names, 2):
            if coint(fw[a], fw[b], trend='c', autolag='aic')[1] < pmax:
                X = np.column_stack([np.ones(form), fw[b]])
                c0, beta = np.linalg.lstsq(X, fw[a], rcond=None)[0]
                e = fw[a] - c0 - beta * fw[b]
                sel.append((a, b, c0, beta, e.mean(), e.std()))
        tw = L.iloc[s0 - 1:s0 + trade]
        idx = tw.index[1:]
        g_w, n_w = np.zeros(len(idx)), np.zeros(len(idx))
        for a, b, c0, beta, mu, sd in sel:
            z = (((tw[a] - c0 - beta * tw[b]) - mu) / sd).values
            ra, rb = R[a].reindex(tw.index).values, R[b].reindex(tw.index).values
            pos, cum = 0, 0.0
            for i in range(1, len(z)):
                r = pos * (ra[i] - beta * rb[i]) / (1 + abs(beta))
                new = pos
                if pos == 0 and z[i] > entry:
                    new = -1
                elif pos == 0 and z[i] < -entry:
                    new = 1
                elif (pos == 1 and z[i] >= 0) or (pos == -1 and z[i] <= 0) or i == len(z) - 1:
                    new = 0
                tc = cost * abs(new - pos)
                if new != 0 and pos == 0:
                    trades += 1
                    cum = 0.0
                cum += r - tc
                if new == 0 and pos != 0:
                    wins += cum > 0
                    pair_ret.append(cum)
                g_w[i - 1] += r / len(sel)
                n_w[i - 1] += (r - tc) / len(sel)
                pos = new
        days.extend(idx)
        gross.extend(g_w)
        net.extend(n_w)
        n_sel.append(len(sel))
    out = pd.DataFrame({'gross': gross, 'net': net}, index=pd.DatetimeIndex(days))
    out = out[~out.index.duplicated()]
    return out, {'windows': len(n_sel), 'pairs_tested': len(names) * (len(names) - 1) // 2,
                 'selected_mean': float(np.mean(n_sel)), 'windows_with_pairs': int(np.sum(np.array(n_sel) > 0)),
                 'trades': int(trades), 'win_rate': float(wins / max(len(pair_ret), 1)),
                 'mean_trade': float(np.mean(pair_ret)) if pair_ret else 0.0}


def perf(r, per_year=252):
    """Annualised mean (%), volatility (%), Sharpe ratio (no risk-free rate) and total log growth (%) of daily returns."""
    m, s = r.mean() * per_year, r.std() * np.sqrt(per_year)
    return {'mean': float(100 * m), 'vol': float(100 * s), 'sharpe': float(m / s) if s > 0 else 0.0,
            'total': float(100 * np.log1p(r).sum()), 'first': str(r.index[0])[:10], 'last': str(r.index[-1])[:10]}


def fig_pair(save_it=True):
    """Banca Transilvania (TLV) and BRD since 2014: log prices; the spread of the full-sample Engle-Granger
    regression as a z-score with the +-2 bands (an in-sample picture: beta, mean and standard deviation use the
    whole sample); the half-life of the spread."""
    P = 100 * np.log(stock_panel(PAIR, PAIR_START))
    r = ols(P.TLV, P.BRD)
    e = pd.Series(r['resid'], index=P.index)
    z = (e - e.mean()) / e.std()
    rho = float(np.polyfit(e.shift(1).dropna().values, e.iloc[1:].values, 1)[0])
    fig, ax = plt.subplots(1, 2, figsize=(9.33, 2.47))
    ax[0].plot(P.index, P.TLV - P.TLV.iloc[0], color=st.IDAred, lw=1.0, label='Banca Transilvania (TLV)')
    ax[0].plot(P.index, P.BRD - P.BRD.iloc[0], color=st.MainBlue, lw=1.0, label='BRD')
    ax[0].set_title('100 ln adjusted price, start = 0')
    years_axis(ax[0], 2)
    ax[1].plot(z.index, z, color=st.Purple, lw=0.8, label='spread z-score (full sample)')
    for b in (-ENTRY, ENTRY):
        ax[1].axhline(b, color=st.IDAred, lw=0.9, ls='--', label='entry bands $\\pm 2$' if b > 0 else '_nolegend_')
    ax[1].axhline(0, color=st.DarkText, lw=0.6)
    ax[1].set_title(f"Spread TLV - {r['b'][1]:.2f} BRD, standardised")
    years_axis(ax[1], 2)
    st.fig_legend_bottom(fig, ncol=4, y=-0.01)
    plt.tight_layout()
    save('tsa_ch7_pair', save_it)
    return {'beta': float(r['b'][1]), 'rho': rho, 'half': half_life(rho), 'first': str(P.index[0])[:10],
            'n': int(len(P)), 'share_out': float(np.mean(np.abs(z) > ENTRY)), 'crossings': int(np.sum(np.diff(np.sign(z.values)) != 0))}


def fig_pairs_backtest(save_it=True):
    """The rolling pairs-trading rule on 8 liquid BVB shares (since 2015) and on 6 US banks (since 2005): cumulative
    log growth gross and net of costs (0.20% and 0.05% per unit of gross exposure traded); the in-sample illusion of
    trading TLV/BRD with full-sample parameters."""
    bvb = stock_panel(BVB, '2015-01-01')
    us = stock_panel(US_BANKS, '2005-01-01', calendar='sp500')
    rb, ib = pairs_backtest(bvb, cost=COST_BVB)
    ru, iu = pairs_backtest(us, cost=COST_US)
    fig, ax = plt.subplots(1, 2, figsize=(9.33, 2.52))
    for a, r, t, cst in [(ax[0], rb, 'Bucharest: 8 shares, 28 pairs', COST_BVB), (ax[1], ru, 'US banks: 6 shares, 15 pairs', COST_US)]:
        a.plot(r.index, 100 * np.log1p(r.gross).cumsum(), color=st.MainBlue, lw=1.2, label='gross of costs')
        a.plot(r.index, 100 * np.log1p(r.net).cumsum(), color=st.IDAred, lw=1.2, label='net of costs')
        a.axhline(0, color=st.DarkText, lw=0.6)
        a.set_title(t + f' (cost {100 * cst:.2f}%)')
        years_axis(a, 2 if a is ax[0] else 4)
    ax[0].set_ylabel('cumulative log growth (%)')
    st.fig_legend_bottom(fig, ncol=2, y=-0.01)
    plt.tight_layout()
    save('tsa_ch7_pairs_backtest', save_it)
    out = {'bvb': {'gross': perf(rb.gross), 'net': perf(rb.net), **ib, 'cost': COST_BVB},
           'us': {'gross': perf(ru.gross), 'net': perf(ru.net), **iu, 'cost': COST_US}}
    # break-even cost: the cost that sets the net mean return to zero (linear in the cost)
    for k, r, info, c in [('bvb', rb, ib, COST_BVB), ('us', ru, iu, COST_US)]:
        drag = (r.gross.mean() - r.net.mean()) / c if c > 0 else np.nan
        out[k]['breakeven'] = float(r.gross.mean() / drag) if drag > 0 else float('nan')
    # the in-sample illusion on TLV/BRD: full-sample beta, mean and sd, the same entry and exit rules
    P = stock_panel(PAIR, PAIR_START)
    L = np.log(P)
    X = np.column_stack([np.ones(len(L)), L.BRD])
    c0, beta = np.linalg.lstsq(X, L.TLV, rcond=None)[0]
    e = L.TLV - c0 - beta * L.BRD
    z = ((e - e.mean()) / e.std()).values
    R = L.diff().values
    pos, rets = 0, []
    for i in range(1, len(z)):
        rets.append(pos * (R[i, 0] - beta * R[i, 1]) / (1 + abs(beta)))
        new = pos
        if pos == 0 and z[i] > ENTRY:
            new = -1
        elif pos == 0 and z[i] < -ENTRY:
            new = 1
        elif (pos == 1 and z[i] >= 0) or (pos == -1 and z[i] <= 0):
            new = 0
        if new != pos:
            rets[-1] -= COST_BVB * abs(new - pos)
        pos = new
    ins = pd.Series(rets, index=P.index[1:])
    out['insample'] = perf(ins)
    out['tlvbrd_rolling'] = pairs_backtest(P, cost=COST_BVB)[1]
    return out

## Run

In [ ]:
print(fig_pair())
print(fig_pairs_backtest())

![tsa_ch7_pair](./tsa_ch7_pair.png)

Output: `tsa_ch7_pair.pdf`

![tsa_ch7_pairs_backtest](./tsa_ch7_pairs_backtest.png)

Output: `tsa_ch7_pairs_backtest.pdf`